<a href="https://colab.research.google.com/github/adityab-tech/PRISMx/blob/main/notebooks/02_Fine_Tuning_Kronos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PRISM - Deliverable 2--Fine-Tuning Kronos on Indian Equity Data

### Objectives

- Load Kronos Foundation Model
- Load Tokenizer
- Load Processed Dataset
- Fine-tune using LoRA
- Evaluate against Zero-Shot
- Save Best Model



#Setup

In [12]:
!git clone https://github.com/shiyu-coder/Kronos.git

fatal: destination path 'Kronos' already exists and is not an empty directory.


In [13]:
!git clone https://github.com/NeoQuasar/Kronos.git

fatal: destination path 'Kronos' already exists and is not an empty directory.


In [14]:
!pip install -q -r requirements.txt

ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


In [15]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
import os
import random
import sys
import warnings
import yaml
import shutil
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from tqdm.auto import tqdm

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

warnings.filterwarnings("ignore")

In [ ]:
!pip -q install transformers peft bitsandbytes accelerate huggingface_hub sentencepiece pyyaml safetensors

In [ ]:
sys.path.append("/content/Kronos")
sys.path.append("/content/Kronos/finetune_csv")

In [ ]:
PROJECT_ROOT = "/content/drive/MyDrive/PRISM"

PROCESSED_PATH = os.path.join(PROJECT_ROOT,"data/processed")
MODEL_PATH = os.path.join(PROJECT_ROOT,"models")

#Fine Tuning

In [ ]:
from model import Kronos, KronosTokenizer
from train_sequential import SequentialTrainer
from config_loader import CustomFinetuneConfig
from finetune_base_model import create_dataloaders

In [ ]:
dataset_path = os.path.join(PROCESSED_PATH,"RELIANCE.NS.csv")
df = pd.read_csv(dataset_path)
df.head()

In [ ]:
df = df.rename(columns={"Date":"timestamps","Open":"open","High":"high","Low":"low","Close":"close","Volume":"volume"})
df["amount"] = 0  #amt is optional
df["timestamps"] = pd.to_datetime(df["timestamps"])
df.head()

In [ ]:
save_path = "/content/Kronos/finetune_csv/data/prism_train.csv"
df.to_csv(save_path,index=False)
print(save_path)

In [ ]:
config = {

"data":{                     #Dataset ko kaise prepare aur split karna hai
"data_path":save_path,
"lookback_window":30,        #Kitna past data input banana hai
"predict_window":5,
"max_context":30,            #Model maximum kitne tokens dekh sakta hai
"clip":5,                    #Values ko safe range me clip karna (outliers control) like yaha pe matlab hai values ko [-5, +5] range ke andar restrict karna hai
"train_ratio":0.8,
"val_ratio":0.2,
"test_ratio":0.0             # kyuki iss fine-tuning phase me sirf training aur validation use ho rahe hain
},

"training":{                 #Training ka behaviour define karta hai
"tokenizer_epochs":2,
"basemodel_epochs":2,
"batch_size":32,
"learning_rate":1e-4,
"num_workers":2
},

"model_paths":{              #Models kahan se load aur kahan save honge
"exp_name":"PRISM",
"pretrained_tokenizer":"NeoQuasar/Kronos-Tokenizer-base", #HF se Kronos tokenizer load karo
"pretrained_predictor":"NeoQuasar/Kronos-base",           #HF se pretrained Kronos model load karo.
"base_save_path":"/content/Kronos/checkpoints"            #Fine-tuned checkpoints isi folder me save honge
}
}

In [ ]:
config_path = "/content/Kronos/finetune_csv/configs/prism.yaml"
with open(config_path,"w") as f:           #Python dictionary (config) ko prism.yaml file me save kar rahe hain, taaki poora project us configuration ko read kar sake
    yaml.dump(config,f)
cfg = CustomFinetuneConfig(config_path)
cfg.print_config_summary()

In [ ]:
#create_dataloaders(cfg) configuration ke basis par train aur validation DataLoaders banata hai,
#aur *_ ka use karke sirf required loaders ko store kiya jata hai jabki baaki returned values ignore kar di jaati hain.
train_loader, val_loader, *_ = create_dataloaders(cfg)

In [ ]:
batch_x, batch_stamp = next(iter(train_loader))
#iter(train_loader) DataLoader ka iterator banata hai aur next() us iterator se pehla batch retrieve karta hai.
#iss batch ko input (batch_x) aur target (batch_stamp) me unpack kiya jata hai

In [ ]:
print(batch_x.shape)
print(batch_stamp.shape)

In [ ]:
tokenizer = KronosTokenizer.from_pretrained("NeoQuasar/Kronos-Tokenizer-base")
model = Kronos.from_pretrained("NeoQuasar/Kronos-base")

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
batch_x = batch_x.to(device)

In [ ]:
with torch.no_grad():
    (
        (reconstruction, encoder_output),    #Tokenizer input ko kitna accurately reconstruct kar pa raha hai & Encoder se nikle continuous features
        bsq_loss,                            #Tokenizer ki quantization/reconstruction training loss
        quantized,                           #Continuous features ka quantized (discrete) representation
        token_ids                            #Final discrete token IDs jo Kronos Transformer consume karega
    ) = tokenizer(batch_x)                   #Tokenizer continuous stock sequences ko discrete token IDs me convert karta hai

In [ ]:
print(reconstruction.shape)
print(encoder_output.shape)
print(quantized.shape)
print(token_ids.shape)
print(bsq_loss)

In [ ]:
print(token_ids.min())
print(token_ids.max())
print(token_ids.unique().numel()) #numel--Tensor me total elements count karta hai

In [ ]:
print(token_ids[:2])

In [ ]:
print("Tokenizer Loss :", bsq_loss.item())

In [ ]:
decoded = tokenizer.decode(token_ids)
print(decoded.shape)

In [ ]:
mse = (( batch_x.cpu()-decoded.cpu())**2).mean()
print(mse.item())

In [ ]:
#does data loading, model initialization, optimization, validation, checkpoint saving, and sequential training phases
trainer = SequentialTrainer(config_path)
print(trainer)

In [ ]:
methods = [m for m in dir(trainer) if "train" in m.lower()]
print(methods)

In [ ]:
print(cfg)
print(cfg.batch_size)
print(cfg.lookback_window)
print(cfg.predict_window)

In [ ]:
cfg.tokenizer_learning_rate = 2e-4
cfg.predictor_learning_rate = 4e-5

In [ ]:
#Kronos Tokenizer ko fine-tune karta hai.Taaki tokenizer stock market data ko better discrete tokens me convert kar sake.
trainer.train_tokenizer_phase()

In [ ]:
print("Tokenizer save path :", cfg.tokenizer_save_path)
print("Basemodel save path :", cfg.basemodel_save_path)
print("Base save path :", cfg.base_save_path)

In [ ]:
print("Tokenizer dir exists:", os.path.exists(cfg.tokenizer_save_path))
print("Basemodel dir exists:", os.path.exists(cfg.basemodel_save_path))

if os.path.exists(cfg.tokenizer_save_path):
    print(os.listdir(cfg.tokenizer_save_path))

In [ ]:
config["model_paths"]["finetuned_tokenizer"] = "/content/Kronos/checkpoints/tokenizer/best_model"
config["model_paths"]["tokenizer_save_name"] = "tokenizer"
config["model_paths"]["basemodel_save_name"] = "basemodel"

In [ ]:
with open(config_path, "w") as f:
    yaml.dump(config, f)
cfg=CustomFinetuneConfig(config_path)

In [ ]:
print("finetuned_tokenizer_path :", cfg.finetuned_tokenizer_path)
print("tokenizer_save_path      :", cfg.tokenizer_save_path)
print("basemodel_save_path      :", cfg.basemodel_save_path)

In [ ]:
trainer = SequentialTrainer(config_path)

In [ ]:
#Fine-tuned tokenizer ko use karke Kronos prediction model train karta hai.Taaki model stock returns ko accurately predict karna seekhe.
trainer.train_basemodel_phase()

In [ ]:
import inspect
source = inspect.getsource(SequentialTrainer.train_basemodel_phase)
print(source)

In [ ]:
print("Tokenizer:")
print(os.listdir("/content/Kronos/checkpoints/tokenizer"))
print("Basemodel:")
print(os.listdir("/content/Kronos/checkpoints/basemodel"))

In [ ]:
print(os.path.exists("/content/Kronos/checkpoints/tokenizer/best_model"))
print(os.path.exists("/content/Kronos/checkpoints/basemodel/best_model"))

In [ ]:
drive_model_dir = os.path.join( PROJECT_ROOT, "models", "kronos")
os.makedirs(drive_model_dir, exist_ok=True)

#shutil.copytree() → Pura folder + uske andar ki saari files copy karta hai
shutil.copytree("/content/Kronos/checkpoints/tokenizer",os.path.join(drive_model_dir, "tokenizer"),dirs_exist_ok=True)
shutil.copytree("/content/Kronos/checkpoints/basemodel",os.path.join(drive_model_dir, "basemodel"),dirs_exist_ok=True)

print("Models copied successfully.")

#LORA

In [ ]:
from model.kronos import Kronos, KronosTokenizer
from peft import LoraConfig, get_peft_model

device = "cuda" if torch.cuda.is_available() else "cpu"
base_model = Kronos.from_pretrained("/content/Kronos/checkpoints/basemodel/best_model").to(device)
base_model.eval()

print(type(base_model))

In [ ]:
#Base model ke saare parameters freeze kardo
for p in base_model.parameters():
    p.requires_grad = False
print("Backbone Frozen!")

In [ ]:
lora_config = LoraConfig(r=16,lora_alpha=32,lora_dropout=0.05,bias="none",  #Effective Scale ≈ alpha / r... alpha is LoRA updates ka scaling factor yeh decide karega ki kitni strength se update karega
    target_modules=["q_proj","k_proj","v_proj","out_proj",])

In [ ]:
!pip install -U torchao

In [ ]:
lora_model = get_peft_model(base_model,lora_config)
lora_model.to(device)

print(type(lora_model))

In [ ]:
lora_model.print_trainable_parameters()

In [ ]:
trainable = []

for name, p in lora_model.named_parameters():
    if p.requires_grad:
        trainable.append(name)

print(len(trainable))
print(trainable[:20])

In [ ]:
tokenizer = KronosTokenizer.from_pretrained(
    "/content/Kronos/checkpoints/tokenizer/best_model"
).to(device)

tokenizer.eval()

print("Tokenizer Loaded!")

In [ ]:
from torch.optim import AdamW
optimizer = AdamW(filter(lambda p: p.requires_grad, lora_model.parameters()),lr=2e-4,weight_decay=1e-2)

print("Optimizer Ready")

In [ ]:
print(lora_model.base_model.model.head.compute_loss)

In [ ]:
batch_x, batch_stamp = next(iter(train_loader))
batch_x = batch_x.to(device)
batch_stamp = batch_stamp.to(device)

with torch.no_grad():
    token_seq_0, token_seq_1 = tokenizer.encode(batch_x,half=True)

In [ ]:
print(token_seq_0.shape)
print(token_seq_1.shape)

In [ ]:
s1_logits, s2_logits = lora_model.base_model.model(token_seq_0,token_seq_1,batch_stamp)

print(s1_logits.shape)
print(s2_logits.shape)

In [ ]:
token_in_0 = token_seq_0[:, :-1]
token_in_1 = token_seq_1[:, :-1]

token_out_0 = token_seq_0[:, 1:]
token_out_1 = token_seq_1[:, 1:]

stamp = batch_stamp[:, :-1]

s1_logits, s2_logits = lora_model.base_model.model(
    token_in_0,
    token_in_1,
    stamp
)

loss, s1_loss, s2_loss = (
    lora_model.base_model.model.head.compute_loss(
        s1_logits,
        s2_logits,
        token_out_0,
        token_out_1
    )
)

print(loss.item())
print(s1_loss.item())
print(s2_loss.item())

In [ ]:
optimizer.zero_grad()
loss.backward()
optimizer.step()

print("One LoRA optimization step completed.")

In [ ]:
for name, p in lora_model.named_parameters():
    if p.requires_grad and p.grad is not None:
        print(name, p.grad.abs().mean().item())

In [ ]:
for name, p in lora_model.named_parameters():
    if p.requires_grad:
        print(name)
        print("Gradient:", p.grad.abs().mean().item())
        break

In [ ]:
lora_model.train()

for step in range(10):

    optimizer.zero_grad()

    with torch.no_grad():
        token_seq_0, token_seq_1 = tokenizer.encode(batch_x, half=True)

    s1_logits, s2_logits = lora_model.base_model.model(
        token_seq_0[:, :-1],
        token_seq_1[:, :-1],
        batch_stamp[:, :-1]
    )

    loss, _, _ = lora_model.base_model.model.head.compute_loss(
        s1_logits,
        s2_logits,
        token_seq_0[:, 1:],
        token_seq_1[:, 1:]
    )

    loss.backward()
    optimizer.step()

    print(f"Step {step+1}: {loss.item():.4f}")

#Evaluation

In [ ]:
from copy import deepcopy

# Zero-shot Kronos
zero_model = Kronos.from_pretrained("NeoQuasar/Kronos-base")
zero_model.eval()

# LoRA trained Kronos
lora_model.eval()

print("Zero-shot loaded")
print("LoRA loaded")

In [ ]:
zero_preds = []
lora_preds = []
targets = []

with torch.no_grad():
    token_seq_0, token_seq_1 = tokenizer.encode(batch_x, half=True)

# Zero-shot
    s1_zero, s2_zero = zero_model(token_seq_0[:, :-1],token_seq_1[:, :-1],batch_stamp[:, :-1] )

#LoRA
    s1_lora, s2_lora = lora_model.base_model.model(token_seq_0[:, :-1],token_seq_1[:, :-1],batch_stamp[:, :-1])
    pred_zero = torch.argmax(s2_zero, dim=-1)
    pred_lora = torch.argmax(s2_lora, dim=-1)

    target = token_seq_1[:, 1:]

print(pred_zero.shape)
print(pred_lora.shape)
print(target.shape)

In [ ]:
zero_acc = (pred_zero == target).float().mean().item()
lora_acc = (pred_lora == target).float().mean().item()

print(f"Zero-shot Accuracy : {100*zero_acc:.2f}%")
print(f"LoRA Accuracy      : {100*lora_acc:.2f}%")

In [ ]:
from scipy.stats import spearmanr
rankics = []

for i in range(pred_lora.size(0)):
    ic, _ = spearmanr(pred_lora[i].cpu().numpy(),target[i].cpu().numpy())

    if not np.isnan(ic):
        rankics.append(ic)

print("Average RankIC :", np.mean(rankics))

In [ ]:
save_path = "/content/Kronos/checkpoints/lora"
os.makedirs(save_path, exist_ok=True)
lora_model.save_pretrained(save_path)
print("LoRA adapters saved!")

In [ ]:
from peft import PeftModel
base_model = Kronos.from_pretrained("/content/Kronos/checkpoints/basemodel/best_model")
loaded_lora = PeftModel.from_pretrained(base_model,"/content/Kronos/checkpoints/lora")

loaded_lora.eval()
print("Reload successful")

In [ ]:
from scipy.stats import spearmanr

all_rankics = []
correct = 0
total = 0
loaded_lora.eval()

with torch.no_grad():
    for batch_x, batch_stamp in val_loader:

        batch_x = batch_x.to(device)
        batch_stamp = batch_stamp.to(device)

        token_seq_0, token_seq_1 = tokenizer.encode(
            batch_x,
            half=True
        )

        s1_logits, s2_logits = loaded_lora.base_model.model(
            token_seq_0[:, :-1],
            token_seq_1[:, :-1],
            batch_stamp[:, :-1]
        )

        pred = torch.argmax(
            s2_logits,
            dim=-1
        )

        target = token_seq_1[:,1:]

        correct += (pred==target).sum().item()
        total += target.numel()

        for i in range(pred.size(0)):

            ic,_ = spearmanr(
                pred[i].cpu().numpy(),
                target[i].cpu().numpy()
            )

            if not np.isnan(ic):
                all_rankics.append(ic)

print("Validation Accuracy :",100*correct/total)
print("Validation RankIC :",np.mean(all_rankics))

In [ ]:
# Evaluation Function

def evaluate_model(model, tokenizer, val_loader, device, is_lora=False):
    model.eval()
    total_correct = 0
    total_tokens = 0
    rankics = []

    with torch.no_grad():
        for batch_x, batch_stamp in val_loader:
            batch_x = batch_x.to(device)
            batch_stamp = batch_stamp.to(device)

            # Encode continuous features
            token_seq_0, token_seq_1 = tokenizer.encode(batch_x, half=True)

            # Forward
            if is_lora:
                s1_logits, s2_logits = model.base_model.model(
                    token_seq_0[:, :-1],
                    token_seq_1[:, :-1],
                    batch_stamp[:, :-1]
                )

            else:
                s1_logits, s2_logits = model(
                    token_seq_0[:, :-1],
                    token_seq_1[:, :-1],
                    batch_stamp[:, :-1]
                )

            # Predictions
            pred = torch.argmax(s2_logits,dim=-1)
            target = token_seq_1[:, 1:]

            # Accuracy
            total_correct += (pred == target).sum().item()
            total_tokens += target.numel()

            #Rank IC
            for i in range(pred.size(0)):

                ic, _ = spearmanr(
                    pred[i].cpu().numpy(),
                    target[i].cpu().numpy()
                )

                if not np.isnan(ic):
                    rankics.append(ic)

    accuracy = 100 * total_correct / total_tokens
    rankic = np.mean(rankics)
    return accuracy, rankic

# Evaluate Zero-shot Kronos
zero_acc, zero_rankic = evaluate_model(zero_model,tokenizer,val_loader,device,is_lora=False)

# Evaluate LoRA Kronos
lora_acc, lora_rankic = evaluate_model( loaded_lora, tokenizer, val_loader, device, is_lora=True)

In [ ]:
print("Zero-shot Accuracy :",zero_acc)
print("LoRA Accuracy :",lora_acc)

print()

print("Zero-shot RankIC :",zero_rankic)
print("LoRA RankIC :",lora_rankic)

In [ ]:
results = pd.DataFrame({
    "Model":["Zero-shot Kronos","LoRA Fine-tuned Kronos"],
    "Accuracy":[zero_acc,lora_acc],
    "RankIC":[zero_rankic,lora_rankic]
})

results